In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.ticker import LogLocator, ScalarFormatter

# 통계분석용
from scipy.stats import mannwhitneyu, kruskal # 최근 6개년 데이터이기 때문에 비모수로 빠집니당 (정규성 검사도 불가능)
import scikit_posthocs as sp # 윌리스... 아니고 월리스 친구

In [ ]:
plt.rcParams.update(plt.rcParamsDefault)

# 커스텀 팔레트
custom_cmap = sns.blend_palette(['#002D72', '#f1f0ec','#ffb659'], as_cmap=True) # light
custom_cmap_dark = sns.blend_palette(['#002D72', '#807266','#ffb659'], as_cmap=True) # dark (가운데만 다름)

# 컬러맵 프리뷰용
data = np.random.randn(10, 10)
sns.heatmap(data, cmap=custom_cmap)
plt.show()

# 그래프 기본 테마 설정
sns.set_theme(style="whitegrid", font_scale=1) # 블루톤

# 그리드 색상 조절
plt.rcParams.update({
    "grid.color": ".8",          # 그리드 색상
    "grid.linestyle": "--",       # 그리드 점선 스타일
    "grid.linewidth": 0.8,        # 그리드 두께
    "axes.grid": True,            # 그리드 항상 켜기
    "axes.edgecolor": ".8",       # 축 테두리 색상
})

# 막대그래프 관련 설정
plt.rcParams.update({
    "lines.linewidth": 2,
    "lines.marker": "D",          # 전역 마커 설정
    "lines.markersize": 7        # 마커 크기
})

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Nanumsquare_ac' # 나눔스퀘어
plt.rcParams['mathtext.fontset'] = 'dejavusans'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 15, 9
plt.rcParams['axes.titlesize'] = 16 # 제목 폰트 사이즈
plt.rcParams['axes.labelsize'] = 14 # 라벨 폰트 사이즈
plt.rcParams['font.size'] = 14 # 기본 폰트사이즈
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['mathtext.fontset'] = 'cm'

In [ ]:
# 알아서 째라...
def get_palette(n):
    return [custom_cmap(i / (n - 1)) for i in range(n)]

# 내가 csv로 가공한것은 다 분석을 위해서였다!

In [ ]:
# 산재
accident_quan = pd.read_csv('data/accident_quantity.csv')
accident_rate = pd.read_csv('data/accident_rate.csv') # rate 붙은거: 사망률 관련 데이터
accident_total_quan = pd.read_csv('data/accident_total_quantity.csv')
accident_total_rate = pd.read_csv('data/accident_total_rate.csv') # 얘도요

In [ ]:
accident_total_quan['지표'].value_counts()

# 액시던트(산업재해)
- 토탈: 전체 업종(세분류 없음)
- 토탈 없는거: 제조업(업종 내 세분류)

## 전체 사업군 내 재해자수 추이

In [ ]:
accident_year = accident_total_quan.query('지표 == "재해자수" and `업종별 중분류` not in "총계"')
accident_year

In [ ]:
sns.lineplot(accident_year, x = '연도', y = '수치', hue = '업종별 중분류', palette="tab20")
plt.title('최근 6개년 업종별 산업재해 재해자 수', y = 1.01)
plt.xlabel('연도')
plt.ylabel('재해자수 (명)')
plt.legend(bbox_to_anchor=(1, 1))
plt.show()

## 전체 사업군 내 사망자 수 추이

In [ ]:
accident_death = accident_total_quan.query('지표 == "사망자수" and `업종별 중분류` not in "총계"')
accident_death

In [ ]:
sns.lineplot(accident_death, x = '연도', y = '수치', hue = '업종별 중분류', palette="tab20")
plt.title('최근 6개년 업종별 산업재해 사망자 수', y = 1.01)
plt.xlabel('연도')
plt.ylabel('사망자수 (명)')
plt.legend(bbox_to_anchor=(1, 1))
plt.show()

- 재해자 수는 제조업이 3위, 사망자 수는 제조업이 2위다. 음... 이거 라인플롯 말고 업종별로 연도 비교해보는건... 아... 안되겠구나. (막대가 너무 많아짐)
- 저거는 전체를 본 거고 TOP 3, TOP 5만 모아서 보는 것도 가능합니다.

## 업종별 평균 비교
- 최근 5개년... 아, 6개년이지.

### 업종별 재해자 수 평균

In [ ]:
accident_omg_mean = accident_year.groupby(['업종별 중분류', '지표']).agg({'수치':'mean'}).reset_index()
accident_omg_mean = accident_omg_mean.sort_values('수치', ascending = False)
# 지표로만 묶었습니당

In [ ]:
ax = sns.barplot(accident_omg_mean, x = '업종별 중분류', y = '수치', hue = '업종별 중분류', palette=get_palette(accident_omg_mean['업종별 중분류'].nunique()))

for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, fmt='%d', padding=3, fontsize=10)

plt.title('업종별 최근 6개년 재해자수 평균', y = 1.01)
plt.xlabel('업종')
plt.ylabel('재해자수 평균 (명)')
plt.yscale('log') # 어업 안보임 이슈로 인하여...
plt.xticks(rotation=45)
plt.show()

### 업종별 사망자 수 평균

In [ ]:
accident_death_mean = accident_death.groupby(['업종별 중분류', '지표']).agg({'수치':'mean'}).reset_index()
accident_death_mean = accident_death_mean.sort_values('수치', ascending = False)

In [ ]:
ax = sns.barplot(accident_death_mean, x = '업종별 중분류', y = '수치', hue = '업종별 중분류', palette=get_palette(accident_death_mean['업종별 중분류'].nunique()))

for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, fmt='%d', padding=3, fontsize=10)

plt.title('업종별 최근 6개년 사망자수 평균', y = 1.01)
plt.xlabel('업종')
plt.ylabel('평균 (명)')
plt.yscale('log')
plt.xticks(rotation=45)
plt.show()

### 한짤요약

In [ ]:
fig, ax = plt.subplots(1, 2)

sns.barplot(accident_omg_mean, x = '업종별 중분류', y = '수치', hue = '업종별 중분류', palette=get_palette(accident_omg_mean['업종별 중분류'].nunique()), ax = ax[0])
sns.barplot(accident_death_mean, x = '업종별 중분류', y = '수치', hue = '업종별 중분류', palette=get_palette(accident_death_mean['업종별 중분류'].nunique()), ax = ax[1])

for a in ax:
    for container in a.containers:
        # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
        a.bar_label(container, fmt='%d', padding=3, fontsize=10)

    # 공통 축 라벨
    a.set_xlabel('업종', fontsize=12, labelpad=10)
    a.set_ylabel('평균 (명)', fontsize=12)

    # 잠시만요 틱좀 눕히고 가실게요
    a.tick_params(axis='x', rotation=90)

    # 스케일 로그 변환(안그러면 막대기가 안보임)
    a.set_yscale('log')

ax[0].set_title('업종별 최근 6개년 재해자수 평균', y = 1.01)
ax[1].set_title('업종별 최근 6개년 사망자수 평균', y = 1.01)

plt.xlabel('업종')
plt.ylabel('평균 (명)')
plt.yscale('log')

plt.tight_layout()
plt.show()

- 제조업, 건설업이 세 손가락 안에 든다.
- 기타의 사업 쟤는 뭐 하는 친구지...?

#### 상자 수염 그림

In [ ]:
# 재해자 수
sns.boxplot(accident_year, x = '업종별 중분류', y='수치', hue = '업종별 중분류', palette = get_palette(accident_year['업종별 중분류'].nunique()))
plt.title('연도별 업종별 산업재해-재해자수', y = 1.01)
plt.xlabel('업종')
plt.ylabel('재해자수 (명)')
plt.yscale('log')

plt.show()

In [ ]:
# 사망자 수
sns.boxplot(accident_death, x = '업종별 중분류', y='수치', hue = '업종별 중분류', palette = get_palette(accident_death['업종별 중분류'].nunique()))
plt.title('연도별 업종별 산업재해-사망자수', y = 1.01)
plt.xlabel('업종')
plt.ylabel('사망자수 (명)')
plt.yscale('log')

plt.show()

### 통계분석

#### 1. 제조업과 건설업 평균간 차이가 통계적으로 유의할까?
- 투트책으로 나눠서 보겠습니다. (재해자 수, 사망자 수)

In [ ]:
# 아 이거 근데 최근 6개년이라서 t-test 못합니다.
# 그럼 어떻게 하냐고요? 비모수로 빠지셔야죠... 괜찮아요 이거 많이 해봤어요... DNA 염기도 비모수로 빠집니다...
manufacture_omg = accident_year.query('`업종별 중분류` == "제조업" and 지표 == "재해자수"') # 제조업 재해자수
manufacture_death = accident_death.query('`업종별 중분류` == "제조업" and 지표 == "사망자수"') # 제조업 사망자수
construct_omg = accident_year.query('`업종별 중분류` == "건설업" and 지표 == "재해자수"') # 건설업 재해자수
construct_death = accident_death.query('`업종별 중분류` == "건설업" and 지표 == "사망자수"') # 건설업 사망자수

- 귀무가설: 제조업과 건설업 간의 재해자수/사망자수 사이에는 통계적으로 유의미한 차이가 존재하지 않는다.
- 대립가설: 제조업과 건설업 간의 재해자수/사망자수 사이에는 통계적으로 유의미한 차이가 존재한다.

In [ ]:
# 재해자수 맨 휫흐니
u_stat, p_value = mannwhitneyu(manufacture_omg['수치'],construct_omg['수치'],alternative="greater")

# 통계량을 보여주세요우
print(f"Mann–Whitney U statistic: {u_stat:.1f}")
if p_value < 0.05:
    print(f'p-value가 {p_value:.2f}이므로 귀무가설을 기각합니다. ')
else:
    print(f'p-value가 {p_value:.2f}이므로 귀무가설을 채택합니다. ')

- 위에 고상하게 썼지만 제조업이나 건설업이나 둘 다 그래프랑 같이 보면 산업재해 많이 터지는걸로는 그 밥에 그 나물이라는 얘기죠.

In [ ]:
# 사망자수 맨 휫흐니
u_stat, p_value = mannwhitneyu(manufacture_death['수치'],construct_death['수치'],alternative="greater")

# 통계량을 보여주세요우
print(f"Mann–Whitney U statistic: {u_stat:.1f}")
if p_value < 0.05:
    print(f'p-value가 {p_value:.2f}이므로 귀무가설을 기각합니다. ')
else:
    print(f'p-value가 {p_value:.2f}이므로 귀무가설을 채택합니다. ')

- 사망자 수도 또이또이 썜쌤이라는 얘기입니다. 

#### 크러스칼-월리스 검정
- 분산분석의 비모수 버전입니다. 따라서 귀무가설도 똑같이 들어갑니다.
- 귀무가설: 업종에 따른 재해자수/사망자수에 차이가 없다.
- 대립가설: 적어도 하나의 업종은 다른 업종과 다를 것이다. 

In [ ]:
# 재해자수
groups = [g['수치'].values for _, g in accident_year.groupby('업종별 중분류')]
stat, p = kruskal(*groups)
print(f"Kruskal-Wallis: stat={stat:.4f}")

# P-value가 유의수준 미만이면 듄 테스트 들어갑니다.
if p < 0.05:
    print(f'p-value가 {p:.2f}이므로 귀무가설을 기각합니다. ')
else:
    print(f'p-value가 {p:.2f}이므로 귀무가설을 채택합니다. ')

In [ ]:
# 사망자수
groups = [g['수치'].values for _, g in accident_death.groupby('업종별 중분류')]
stat, p = kruskal(*groups)
print(f"Kruskal-Wallis: stat={stat:.4f}")

# P-value가 유의수준 미만이면 듄 테스트 들어갑니다.
if p < 0.05:
    print(f'p-value가 {p:.2f}이므로 귀무가설을 기각합니다. ')
else:
    print(f'p-value가 {p:.2f}이므로 귀무가설을 채택합니다. ')

#### Dunn's test
- ANOVA 하면 항상 Tukey가 따라오는데, 크러스칼-월리스 검정에도 비슷한 친구가 하나 있습니다.
- 저게 근데 듄이냐 던이냐...

In [ ]:
# 재해자 수
dunn_result = sp.posthoc_dunn(accident_year, val_col='수치', group_col='업종별 중분류', p_adjust='bonferroni')
print(dunn_result)

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(
    dunn_result,
    annot=True,
    fmt='.3f',
    cmap='RdYlGn',  # 낮을수록(유의할수록) 빨강
    vmin=0, vmax=0.05,  # 0.05 기준으로 색상
    linewidths=0.5,
    square=True
)
plt.title("Dunn's Test p-value 히트맵 (Bonferroni 보정)\n재해자수 업종 간 비교", fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

- 해석하는 법
1. 당황하지 마세요, 이건 저 위에 있는 텍스트를 보기 좋기 히트맵으로 그린 겁니다. 군데군데 보이는 빨간색이 자세히 보니까 '얘랑 얘랑 다르네?'입니다.
2. 근데 박스플롯이랑 결과가 다르죠? 이건 왜 그러냐면 저게 최근 6개년 평균이라서 그런겁니다. 그러니까 저기 빨간색은 확실히 차이가 있음! 이고 초록색은 **통계적으로 유의하지 아니하지만 차이가 없다고 딱 말하기도 애매한** 상황입니다.

In [ ]:
# 사망자 수
dunn_result = sp.posthoc_dunn(accident_death, val_col='수치', group_col='업종별 중분류', p_adjust='bonferroni')
print(dunn_result)

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(
    dunn_result,
    annot=True,
    fmt='.3f',
    cmap='RdYlGn',  # 낮을수록(유의할수록) 빨강
    vmin=0, vmax=0.05,  # 0.05 기준으로 색상
    linewidths=0.5,
    square=True
)
plt.title("Dunn's Test p-value 히트맵 (Bonferroni 보정)\n사망자수 업종 간 비교", fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

1. 얘도 마찬가지로 저기 빨간색은 확실히 차이가 있음! 이고 초록색은 **통계적으로 유의하지 아니하지만 차이가 없다고 딱 말하기도 애매한** 상황입니다. 이거 써먹기가 대단히 애매하네요.

# 재해율/사망만인율
- 그... 퍼밀 아시죠 퍼밀? ‰로 쓰는건데 이게 1/1000을 의미합니다. 바닷물 염분 이런거 달 때 나오는거예요.
- 그리고 사망만인율은 단위로 ‱, 퍼밀리아드를 씁니다. 이 기호는 1/10000을 의미해요. 임금근로자 1만 명당 발생하는 사고사망자 수의 비율입니다.
- 백분율은 100분의 얼마, 천분율(퍼밀)은 천분의 얼마, 만분율(퍼밀리아드)은 만분의 얼마입니다.

In [ ]:
accident_total_rate

## 업종별 재해율, 사망만인율 추이

In [ ]:
# 재해율
accident_disaster = accident_total_rate.query('지표 == "재해율" and `업종별 중분류` not in "총계"')

# 사망만인율
accident_permyriad = accident_total_rate.query('지표 == "사망만인율" and `업종별 중분류` not in "총계"')

In [ ]:
sns.lineplot(accident_disaster, x = '연도', y = '수치', hue = '업종별 중분류', palette=get_palette(accident_disaster['업종별 중분류'].nunique()))
plt.title('최근 6개년 재해율 추이', y = 1.01)
plt.xlabel('연도')
plt.ylabel('재해율 (%)')
plt.legend(bbox_to_anchor=(1, 1))
plt.show()

In [ ]:
sns.lineplot(accident_permyriad, x = '연도', y = '수치', hue = '업종별 중분류', palette=get_palette(accident_permyriad['업종별 중분류'].nunique()))
plt.title('최근 6개년 사망만인율 추이', y = 1.01)
plt.xlabel('연도')
plt.ylabel('사망만인율 (만분율)')
plt.legend(bbox_to_anchor=(1, 1))
plt.show()

- 재해율, 사망만인율은 광업이 제일 높고 나머지는 고만고만하네요.

## 업종별 최근 6개년 평균

In [ ]:
# 재해율
accident_disaster_mean = accident_disaster.groupby(['업종별 중분류','지표']).agg({'수치':'mean'}).reset_index()
accident_disaster_mean = accident_disaster_mean.sort_values('수치', ascending = False)

# 사망만인율
accident_permyriad_mean = accident_permyriad.groupby(['업종별 중분류','지표']).agg({'수치':'mean'}).reset_index()
accident_permyriad_mean = accident_permyriad_mean.sort_values('수치', ascending = False)

In [ ]:
ax = sns.barplot(accident_disaster_mean, x = '업종별 중분류', y = '수치', hue = '업종별 중분류', palette=get_palette(accident_disaster_mean['업종별 중분류'].nunique()))

for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10)

plt.title('업종별 최근 6개년 재해율 평균', y = 1.01)
plt.xlabel('업종')
plt.ylabel('평균 재해율 (%)')
plt.yscale('log') # Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol. (마이너스 어짜고 줬는디...)

# 상기 이유로 이게 최선이었습니다. (로그축 안해도 되면 빼도 무관함)
ax.yaxis.set_major_locator(LogLocator(base=10))
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.yaxis.get_major_formatter().set_scientific(False)

plt.xticks(rotation=45)
plt.show()

In [ ]:
ax = sns.barplot(accident_permyriad_mean, x = '업종별 중분류', y = '수치', hue = '업종별 중분류', palette=get_palette(accident_permyriad_mean['업종별 중분류'].nunique()))

for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10)

plt.title('업종별 최근 6개년 사망만인율 평균', y = 1.01)
plt.xlabel('업종')
plt.ylabel('평균 사망만인율 (만분율)')
plt.yscale('log')
plt.xticks(rotation=45)
plt.show()

# 제조업 발골작업

In [ ]:
manu_quan = accident_total_quan.query('`업종별 중분류` == "제조업"')

In [ ]:
manu_rate = accident_total_rate.query('`업종별 중분류` == "제조업"')
manu_rate

## 최근 6년간 추이
- 재해자수, 사망자수, 재해율, 사망만인율 봅니다.

In [ ]:
manu_accident = manu_quan.query('지표 == "재해자수"') # 재해자수
manu_death = manu_quan.query('지표 == "사망자수"') # 사망자수
manu_accident_rate = manu_rate.query('지표 == "재해율"') # 재해율
manu_death_permyriad = manu_rate.query('지표 == "사망만인율"') # 사망만인율

### 따로(숫자)

In [ ]:
sns.lineplot(manu_accident, x = '연도', y = '수치', color=custom_cmap(0.1))
plt.title('최근 6개년 재해자수 추이 (제조업)', y = 1.01)
plt.xlabel('연도')
plt.ylabel('재해자수 (명)')
plt.ticklabel_format(axis='y', style='plain')
plt.xticks([2019, 2020, 2021, 2022, 2023, 2024]) # 얘는 왜 이걸 해줘야 년도로 나올까...
plt.show()

- 이거... 계단 아닙니까 계단... 

In [ ]:
sns.lineplot(manu_death, x = '연도', y = '수치', color=custom_cmap(0.9))
plt.title('최근 6개년 사망자수 추이 (제조업)', y = 1.01)
plt.xlabel('연도')
plt.ylabel('사망자수 (명)')
plt.ticklabel_format(axis='y', style='plain')
plt.xticks([2019, 2020, 2021, 2022, 2023, 2024]) # 얘는 왜 이걸 해줘야 년도로 나올까...
plt.show()

### 따로(비율)

In [ ]:
sns.lineplot(manu_accident_rate, x = '연도', y = '수치', color=custom_cmap(0.1))
plt.title('최근 6개년 재해율 추이 (제조업)', y = 1.01)
plt.xlabel('연도')
plt.ylabel('재해율(%)')
plt.ticklabel_format(axis='y', style='plain')
plt.xticks([2019, 2020, 2021, 2022, 2023, 2024]) # 얘는 왜 이걸 해줘야 년도로 나올까...
plt.show()

In [ ]:
sns.lineplot(manu_death_permyriad, x = '연도', y = '수치', color=custom_cmap(0.9))
plt.title('최근 6개년 사망만인율 추이 (제조업)', y = 1.01)
plt.xlabel('연도')
plt.ylabel('사망만인율 (만분율)')
plt.ticklabel_format(axis='y', style='plain')
plt.xticks([2019, 2020, 2021, 2022, 2023, 2024]) # 얘는 왜 이걸 해줘야 년도로 나올까...
plt.show()

- 사망자율과 사망만인율은 2021년에 피크를 찍고 점차 하락세인데, 재해율은 올라가고 있습니다.
- 산업재해때문에 부상을 입는 사람은 많아도 사망자는 줄어들었다...는 얘기겠죠?

### 같이보기 (재해 관련 지표들)

In [ ]:
fig, ax = plt.subplots(1, 2)

ax[0] = sns.lineplot(manu_accident, x = '연도', y = '수치', color=custom_cmap(0.1), ax = ax[0])
ax[1] = sns.lineplot(manu_accident_rate, x = '연도', y = '수치', color=custom_cmap(0.1), ax = ax[1])

ax[0].set_title('최근 6개년 재해자수 추이 (제조업)', y = 1.01)
ax[1].set_title('최근 6개년 재해율 추이 (제조업)', y = 1.01)

plt.tight_layout()
plt.show()

#### 이중 축 그래프

In [ ]:
fig, ax = plt.subplots(figsize=(15, 9))

ax.plot(manu_accident['연도'], manu_accident['수치'], color=custom_cmap(0.1), alpha=0.8, label="재해자수")
ax.set_ylabel('재해자수 (명)', fontsize=12)
ax.set_xlabel('연도', fontsize=12)
ax.set_ylim(0, manu_accident['수치'].max() * 1.2)
ax.grid(axis='y', linestyle='--', alpha=0.7)  # 주축 그리드만 켜기

ax1 = ax.twinx()
ax1.plot(manu_accident_rate['연도'], manu_accident_rate['수치'],color=custom_cmap(0.9), marker='o', linewidth=2, label="재해율")
ax1.set_ylabel('재해율 (%)', fontsize=12)
ax1.set_ylim(
    manu_accident_rate['수치'].min() * 0.95,
    manu_accident_rate['수치'].max() * 1.05
)
ax1.grid(False)  # 보조축 그리드 끄기 ✓

ax.set_title('최근 6개년 재해자수 및 재해율 추이 (제조업)', y=1.01)
ax.set_xticks(manu_accident['연도'])

# 두 축의 핸들 합치기
handles1, labels1 = ax.get_legend_handles_labels()   # 막대
handles2, labels2 = ax1.get_legend_handles_labels()  # 선

ax.legend(handles1 + handles2, labels1 + labels2, loc='upper left', fontsize=12)

plt.tight_layout()
plt.show()

### 같이보기 (사망 관련 지표들)

In [ ]:
fig, ax = plt.subplots(1, 2)

ax[0] = sns.lineplot(manu_death, x = '연도', y = '수치', color=custom_cmap(0.9), ax = ax[0])
ax[1] = sns.lineplot(manu_death_permyriad, x = '연도', y = '수치', color=custom_cmap(0.9), ax = ax[1])

ax[0].set_title('최근 6개년 사망자수 추이 (제조업)', y = 1.01)
ax[1].set_title('최근 6개년 사망만인율 추이 (제조업)', y = 1.01)

plt.tight_layout()
plt.show()

#### 이중 축 그래프

In [ ]:
fig, ax = plt.subplots(figsize=(15, 9))

ax.plot(manu_death['연도'], manu_death['수치'], color=custom_cmap(0.1), alpha=0.8, label="사망자수")
ax.set_ylabel('사망자수 (명)', fontsize=12)
ax.set_xlabel('연도', fontsize=12)
ax.set_ylim(0, manu_death['수치'].max() * 1.2)
ax.grid(axis='y', linestyle='--', alpha=0.7)  # 주축 그리드만 켜기

ax1 = ax.twinx()
ax1.plot(manu_death_permyriad['연도'], manu_death_permyriad['수치'], color=custom_cmap(0.9), marker='o', linewidth=2, label="사망만인율")
ax1.set_ylabel('사망만인율 (만분율)', fontsize=12)
ax1.set_ylim(
    manu_death_permyriad['수치'].min() * 0.95,
    manu_death_permyriad['수치'].max() * 1.05
)
ax1.grid(False)  # 보조축 그리드 끄기 ✓

ax.set_title('최근 6개년 사망자수 및 사망만인율 추이 (제조업)', y=1.01)
ax.set_xticks(manu_death['연도'])

# 두 축의 핸들 합치기
handles1, labels1 = ax.get_legend_handles_labels()   # 막대
handles2, labels2 = ax1.get_legend_handles_labels()  # 선

ax.legend(handles1 + handles2, labels1 + labels2, loc='upper left', fontsize=12)

plt.tight_layout()
plt.show()

- 사망만인율이 왜이렇게 들쭉날쭉하죠? 아니 사망자 수는 그대로잖아요.
- 자, 우리가 이 수수께끼에 대답하기 위해서 필요한 게 근로자 수입니다. 뭔 소리여? 자자 들어보세요. 지가르데라는 포켓몬이 있습니다. 이 포켓몬은 10%, 50%, 퍼펙트(100%)폼이 있고 퍼펙트폼에서 메가진화를 하는데 전투중에 퍼펙트폼으로 폼 체인지를 할 때마다 체력이 회복됩니다. 예? 아니 그게 돼요?
- 사실 체력을 실제로 회복하는 게 아니라, HP의 최댓값이 바뀌어서 그렇게 된 겁니다. 들어보세요, HP가 100일때 60의 대미지를 입는 건 어 씁 회복각인데?지만 HP가 200일때 60의 대미지를 입는 건 음 쫌 아픈데? 수준이잖아요? 그런겁니다.

#### 근로자 수 대비 사망만인율

In [ ]:
manu_people = manu_quan.query('지표 == "근로자수"')
manu_people

In [ ]:
fig, ax = plt.subplots(figsize=(15, 9))

ax.plot(manu_people['연도'], manu_people['수치'], color=custom_cmap(0.1), alpha=0.8, label="근로자수")
ax.set_ylabel('근로자수 (명)', fontsize=12)
ax.set_xlabel('연도', fontsize=12)
ax.set_ylim(0, manu_people['수치'].max() * 1.2)
ax.grid(axis='y', linestyle='--', alpha=0.7)  # 주축 그리드만 켜기

ax1 = ax.twinx()
ax1.plot(manu_death_permyriad['연도'], manu_death_permyriad['수치'], color=custom_cmap(0.9), marker='o', linewidth=2, label="사망만인율")
ax1.set_ylabel('사망만인율 (만분율)', fontsize=12)
ax1.set_ylim(
    manu_death_permyriad['수치'].min() * 0.95,
    manu_death_permyriad['수치'].max() * 1.05
)
ax1.grid(False)  # 보조축 그리드 끄기 ✓

ax.set_title('최근 6개년 근로자수 및 사망만인율 추이 (제조업)', y=1.01)
ax.set_xticks(manu_people['연도'])
ax.ticklabel_format(axis='y', style='plain')

# 두 축의 핸들 합치기
handles1, labels1 = ax.get_legend_handles_labels()   # 막대
handles2, labels2 = ax1.get_legend_handles_labels()  # 선

ax.legend(handles1 + handles2, labels1 + labels2, loc='upper left', fontsize=12)

plt.tight_layout()
plt.show()

- 수수께끼는 풀렸다! 저 축이 미미해서 잘 안보이실수도 있는데, 2021년에 사망자 수는 증가하고 근로자 수는 줄어들어서 사망만인율이 훅 뛰게 된 겁니다. 그러니까 아까 그 게임 비유로 보자면
1. 받은 대미지도 치명타였는데
2. 내 최대 HP까지 깎인

상황인거죠.

## 얼마나 변했나?

### 재해율, 재해자수 변화

In [ ]:
manu_accident = manu_quan.query('지표 == "재해자수"') # 재해자수
manu_death = manu_quan.query('지표 == "사망자수"') # 사망자수
manu_accident_rate = manu_rate.query('지표 == "재해율"') # 재해율
manu_death_permyriad = manu_rate.query('지표 == "사망만인율"') # 사망만인율

 #### 재해자 수 및 재해율 변화량
- 재해자 수: pct_chage() <<변화율
- 재해율: diff() <<단순 변화량
- 우리 이거 밑에 사망자수랑 사망만인율에도 적용해야돼요

In [ ]:
# 변화율
manu_accident = manu_accident.copy()
manu_death = manu_death.copy()

manu_accident['변화'] = manu_accident['수치'].pct_change() * 100
manu_death['변화'] = manu_death['수치'].pct_change() * 100

In [ ]:
# 변화량
manu_accident_rate = manu_accident_rate.copy()
manu_death_permyriad = manu_death_permyriad.copy()

manu_accident_rate['변화'] = manu_accident_rate['수치'].diff()
manu_death_permyriad['변화'] = manu_death_permyriad['수치'].diff()

#### 파이널 퓨-전-

In [ ]:
accident_cnt = pd.concat([manu_accident, manu_accident_rate])

accident_cnt

In [ ]:
# 근데 이거 단위도 같은데 걍 같이 그리면 안되나...
ax = sns.barplot(data=accident_cnt.dropna(), x='연도', y='변화', hue='지표', palette=get_palette(2))

max_abs = max(abs(accident_cnt['변화'].min()), abs(accident_cnt['변화'].max())) * 1.2
ax.set_ylim(-max_abs, max_abs)
# 0% 기준선 (이보다 위면 상승, 아래면 하락)
plt.axhline(0, color='black', linewidth=1, alpha=0.5)

for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10)

plt.title('제조업 재해 지표 전년 대비 변동', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# 1. 데이터 분리
data1 = accident_cnt[accident_cnt['지표'] == '재해자수']
data2 = accident_cnt[accident_cnt['지표'] == '재해율']

fig, ax1 = plt.subplots(figsize=(15, 9))
ax1.grid(True, axis='y', linestyle='--', alpha=0.3)
ax1.grid(False, axis='x') # 세로선은 확실히 제거
ax2 = ax1.twinx()
ax2.grid(False)

# 막대 너비 설정
width = 0.35
x = np.arange(len(data1['연도']))

# 2. 왼쪽 축: 재해자수 (위치를 왼쪽으로 살짝 밀기)
bar1 = ax1.bar(x - width/2, data1['변화'], width, label='재해자수(%)', color=custom_cmap(0.1))

# 3. 오른쪽 축: 재해율 (위치를 오른쪽으로 살짝 밀기)
bar2 = ax2.bar(x + width/2, data2['변화'], width, label='재해율(p)', color=custom_cmap(0.9))

# [핵심] 영점(0) 맞추기
# 양쪽 축의 데이터 범위를 계산해서 0점이 같은 높이에 오도록 스케일을 조정합니다.
def align_zeros(ax_left, ax_right):
    l_min, l_max = ax_left.get_ylim()
    r_min, r_max = ax_right.get_ylim()

    # 더 큰 비율의 범위를 찾아 양쪽 축의 비율을 통일시킴
    l_ratio = l_max / abs(l_min) if l_min != 0 else l_max
    r_ratio = r_max / abs(r_min) if r_min != 0 else r_max
    # (실제 구현 시에는 더 복잡한 계산이 필요하지만, 대칭 범위를 쓰는 게 가장 깔끔합니다)

    # 가장 확실한 방법: 양쪽 다 대칭 범위로 강제 설정
    l_limit = max(abs(l_min), abs(l_max)) * 1.2
    r_limit = max(abs(r_min), abs(r_max)) * 1.2
    ax_left.set_ylim(-l_limit, l_limit)
    ax_right.set_ylim(-r_limit, r_limit)

align_zeros(ax1, ax2)

# 4. 마무리 및 라벨링
ax1.set_xticks(x)
ax1.set_xticklabels(data1['연도'])
ax1.axhline(0, color='black', linewidth=1.5)

# 값 표시 (각 막대 위치에 맞게)
ax1.bar_label(bar1, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')
ax2.bar_label(bar2, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

ax1.set_ylabel('재해자수 변화율 (%)')
ax2.set_ylabel('재해울 변화 (p)')

ax1.set_xlabel('연도')
plt.title('제조업 재해 지표 전년 대비 변동', pad=20)
plt.show()

### 사망율, 사망만인율 변화

In [ ]:
# 아 이걸 가오가이거 들으면서 했어야 하는건데 아
death_cnt = pd.concat([manu_death, manu_death_permyriad])

death_cnt

In [ ]:
# 근데 이거 단위도 같은데 걍 같이 그리면 안되나...
ax = sns.barplot(data=death_cnt.dropna(), x='연도', y='변화', hue='지표', palette=get_palette(2))

max_abs = max(abs(death_cnt['변화'].min()), abs(death_cnt['변화'].max())) * 1.2
ax.set_ylim(-max_abs, max_abs)
# 0% 기준선 (이보다 위면 상승, 아래면 하락)
plt.axhline(0, color='black', linewidth=1, alpha=0.5)

for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10)

plt.title('제조업 재해 지표 전년 대비 변동', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# 1. 데이터 분리
data1 = death_cnt[death_cnt['지표'] == '사망자수']
data2 = death_cnt[death_cnt['지표'] == '사망만인율']

fig, ax1 = plt.subplots(figsize=(15, 9))
ax1.grid(True, axis='y', linestyle='--', alpha=0.3)
ax1.grid(False, axis='x') # 세로선은 확실히 제거
ax2 = ax1.twinx()
ax2.grid(False)

# 막대 너비 설정
width = 0.35
x = np.arange(len(data1['연도']))

# 2. 왼쪽 축: 재해자수 (위치를 왼쪽으로 살짝 밀기)
bar1 = ax1.bar(x - width/2, data1['변화'], width, label='사망자수(%)', color=custom_cmap(0.1))

# 3. 오른쪽 축: 재해율 (위치를 오른쪽으로 살짝 밀기)
bar2 = ax2.bar(x + width/2, data2['변화'], width, label='사망만인율(p)', color=custom_cmap(0.9))

# [핵심] 영점(0) 맞추기
# 양쪽 축의 데이터 범위를 계산해서 0점이 같은 높이에 오도록 스케일을 조정합니다.
def align_zeros(ax_left, ax_right):
    l_min, l_max = ax_left.get_ylim()
    r_min, r_max = ax_right.get_ylim()

    # 더 큰 비율의 범위를 찾아 양쪽 축의 비율을 통일시킴
    l_ratio = l_max / abs(l_min) if l_min != 0 else l_max
    r_ratio = r_max / abs(r_min) if r_min != 0 else r_max
    # (실제 구현 시에는 더 복잡한 계산이 필요하지만, 대칭 범위를 쓰는 게 가장 깔끔합니다)

    # 가장 확실한 방법: 양쪽 다 대칭 범위로 강제 설정
    l_limit = max(abs(l_min), abs(l_max)) * 1.2
    r_limit = max(abs(r_min), abs(r_max)) * 1.2
    ax_left.set_ylim(-l_limit, l_limit)
    ax_right.set_ylim(-r_limit, r_limit)

align_zeros(ax1, ax2)

# 4. 마무리 및 라벨링
ax1.set_xticks(x)
ax1.set_xticklabels(data1['연도'])
ax1.axhline(0, color='black', linewidth=1.5)

# 값 표시 (각 막대 위치에 맞게)
ax1.bar_label(bar1, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')
ax2.bar_label(bar2, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

ax1.set_ylabel('사망자수 변화율 (%)')
ax2.set_ylabel('사망만인율 변화 (p)')

ax1.set_xlabel('연도')
plt.title('제조업 사망 관련 지표 전년 대비 변동', pad=20)
plt.show()

#### 근로자수, 사망자수, 사망만인율

In [ ]:
# 근로자 수 변화율
manu_people = manu_people.copy()
manu_people['변화'] = manu_people['수치'].pct_change() * 100

death_people_filter = pd.concat([manu_death, manu_death_permyriad, manu_people], axis = 0)
death_people_filter

In [ ]:
# 3. 그래프 시각화 (첫 해인 2020년은 데이터가 NaN이므로 2021년부터 표시됨)
plt.figure(figsize=(15, 9))
ax = sns.barplot(data=death_people_filter.dropna(), x='연도', y='변화', hue='지표', palette=get_palette(3))

for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10)
# 0% 기준선 (이보다 위면 상승, 아래면 하락)
plt.axhline(0, color='black', linewidth=1)

plt.title('제조업 근로자수 및 사망 관련 지표 (전년 대비 변화)', y=1.05)
plt.ylabel('변화량')
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.legend(title='급여 항목', fontsize=12)
plt.show()

- 와... 이건 이중축으로도 안되겠는데...
- 사실 단위가 달라서 축을 빼야 하긴 해요... 사망만인율은 %가 아니라 만분율이거든요. 